# ImprovedGS thuan: fine-tune doc lap cho tung test pose

Moi test pose load lai model 30k, cham tat ca train view bang
`exp(-d^2 / (2 * (3 * sigma)^2)) * max(0, cos(theta))^2`, chon top-25,
roi train dung 3.000 optimizer update. C2F va pose-aware sampling deu tat;
densify/split chi chay trong 1.500 step dau. Batch dau chay dung 15 pose
dau tien theo thu tu `test_poses.csv`, khong random.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile

PHASE_DIR = Path('/kaggle/input/datasets/xuanph/phase1/phase1')
WORK_ROOT = Path('/kaggle/working')
REPO_DIR = WORK_ROOT / 'Improved-GS'
REPO_BRANCH = 'agent/test-pose-finetune'

# ========== CHI CAN DOI POSE_BATCH_INDEX CHO CAC LAN SAU ==========
SET_NAME = 'public_set'
SCENE_NAME = 'HCM0204'
POSE_BATCH_INDEX = 0  # 0: pose 0-14, 1: pose 15-29, 2: pose 30-44, ...
POSE_BATCH_SIZE = 15

if POSE_BATCH_INDEX < 0 or POSE_BATCH_SIZE < 1:
    raise ValueError('POSE_BATCH_INDEX phai >= 0 va POSE_BATCH_SIZE phai >= 1')
POSE_START_INDEX = POSE_BATCH_INDEX * POSE_BATCH_SIZE
POSE_COUNT = POSE_BATCH_SIZE
POSE_END_INDEX = POSE_START_INDEX + POSE_COUNT

BASE_ITERATION = 30_000
FINE_TUNE_STEPS = 3_000
SPLIT_UNTIL_STEP = 1_500
TOP_K = 25
SIGMA_MULTIPLIER = 3.0
DENSIFY_GRAD_THRESHOLD = 0.0002
SAVE_POSE_MODELS = False
SAVE_PNG = True

DATA_SET = PHASE_DIR / SET_NAME
DATA_ROOT = WORK_ROOT / 'vai_cleaned' / SET_NAME
SOURCE_PATH = DATA_ROOT / SCENE_NAME
EXPERIMENT_NAME = (
    f'test_pose_finetune_imgs_main_batch{POSE_BATCH_INDEX:02d}_'
    f'poses{POSE_START_INDEX:03d}_{POSE_END_INDEX - 1:03d}'
)
OUTPUT_MODEL_ROOT = WORK_ROOT / 'vai_models' / EXPERIMENT_NAME / SET_NAME / SCENE_NAME
RENDER_ROOT = WORK_ROOT / 'vai_renders' / EXPERIMENT_NAME / SET_NAME
PNG_ROOT = WORK_ROOT / 'vai_png' / EXPERIMENT_NAME / SET_NAME

# Neu Kaggle input co nhieu model 30k cua cung scene, dat duong dan thu cong tai day.
BASE_MODEL_OVERRIDE = ''
if BASE_MODEL_OVERRIDE:
    BASE_MODEL = Path(BASE_MODEL_OVERRIDE)
else:
    base_ply_candidates = sorted(
        path for path in Path('/kaggle/input').glob(
            f'**/{SCENE_NAME}/point_cloud/iteration_{BASE_ITERATION}/point_cloud.ply'
        )
        if path.is_file()
    )
    base_model_candidates = sorted({path.parents[2] for path in base_ply_candidates})
    if len(base_model_candidates) != 1:
        raise RuntimeError(
            'Can dung 1 ImprovedGS main model 30k. '
            f'Tim duoc: {base_model_candidates}. Hay dat BASE_MODEL_OVERRIDE.'
        )
    BASE_MODEL = base_model_candidates[0]

print('Experiment:', EXPERIMENT_NAME)
print('Pose indices:', POSE_START_INDEX, 'to', POSE_END_INDEX - 1)
print('Base model:', BASE_MODEL)

In [ ]:
# Clone dung nhanh thi nghiem moi; neu da co repo thi cap nhat nhanh.
if not REPO_DIR.exists():
    subprocess.run([
        'git', 'clone', '--recursive', '--branch', REPO_BRANCH,
        'https://github.com/mdd206/Improved-GS.git', str(REPO_DIR),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_BRANCH], check=True)
    subprocess.run([
        'git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', REPO_BRANCH,
    ], check=True)
os.chdir(REPO_DIR)
assert (REPO_DIR / 'vai_test_pose_finetune.py').is_file(), REPO_DIR
print('Repo:', Path.cwd())

In [ ]:
# Cai COLMAP cho pipeline main: SIMPLE_RADIAL -> PINHOLE RGBA.
def install_colmap_with_apt():
    apt_env = os.environ.copy()
    apt_env['DEBIAN_FRONTEND'] = 'noninteractive'
    options = [
        '-o', 'Dpkg::Use-Pty=0',
        '-o', 'Dpkg::Lock::Timeout=120',
        '-o', 'Acquire::Retries=3',
    ]
    print('Bat dau apt-get update...', flush=True)
    update_result = subprocess.run(
        ['apt-get', *options, 'update'], check=False, env=apt_env,
    )
    if update_result.returncode != 0:
        return False
    print('Bat dau cai COLMAP...', flush=True)
    install_result = subprocess.run([
        'apt-get', *options, 'install', '-y', '--no-install-recommends', 'colmap',
    ], check=False, env=apt_env)
    return install_result.returncode == 0


def install_colmap_with_conda():
    conda_cli = next(
        (name for name in ('micromamba', 'mamba', 'conda') if shutil.which(name)),
        None,
    )
    if conda_cli is None:
        return False
    colmap_env = WORK_ROOT / 'colmap-env'
    action = 'install' if (colmap_env / 'conda-meta').is_dir() else 'create'
    result = subprocess.run([
        conda_cli, action, '-y', '-p', str(colmap_env), '-c', 'conda-forge', 'colmap',
    ], check=False)
    colmap_bin = colmap_env / 'bin' / 'colmap'
    if result.returncode == 0 and colmap_bin.is_file():
        os.environ['PATH'] = str(colmap_bin.parent) + os.pathsep + os.environ.get('PATH', '')
        return True
    return False


if shutil.which('colmap') is None and not install_colmap_with_apt():
    print('APT khong cai duoc COLMAP, thu Conda fallback.', flush=True)
if shutil.which('colmap') is None:
    install_colmap_with_conda()
if shutil.which('colmap') is None:
    raise RuntimeError('Khong the cai COLMAP CLI')
print('COLMAP:', shutil.which('colmap'))

# Cai dependency Python va build CUDA extension voi log ro rang.
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'numpy==1.26.1', 'opencv-python==4.10.0.82',
    'setuptools==69.5.1', 'ninja', 'tqdm', 'plyfile',
], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Hay bat GPU Accelerator tren Kaggle')
cuda_major, cuda_minor = torch.cuda.get_device_capability()
build_env = os.environ.copy()
build_env['MAX_JOBS'] = '2'
build_env['TORCH_CUDA_ARCH_LIST'] = f'{cuda_major}.{cuda_minor}'
print('Torch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0), 'arch:', build_env['TORCH_CUDA_ARCH_LIST'])
for package_dir in [
    'submodules/diff-gaussian-rasterization',
    'submodules/simple-knn',
    'submodules/fused-ssim',
]:
    print('Bat dau build:', package_dir, flush=True)
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-v',
        '--no-build-isolation', '--no-deps', package_dir,
    ], check=True, env=build_env, timeout=1200)
    print('Build xong:', package_dir, flush=True)
print('Done', flush=True)

In [ ]:
# Pipeline main: undistort SIMPLE_RADIAL thanh PINHOLE RGBA.
if not DATA_SET.is_dir():
    raise FileNotFoundError(DATA_SET)
subprocess.run([
    sys.executable, '-u', 'vai_preprocess.py',
    '--input', str(DATA_SET),
    '--output', str(DATA_ROOT),
    '--subset', SCENE_NAME,
    '--overwrite',
], check=True)
print('Preprocess xong:', SOURCE_PATH)

In [ ]:
# Chan dung nham D3, C2F, pose-aware hoac model khong phai ImprovedGS.
base_ply = BASE_MODEL / 'point_cloud' / f'iteration_{BASE_ITERATION}' / 'point_cloud.ply'
if not base_ply.is_file():
    raise FileNotFoundError(base_ply)
if 'native_simple_radial' in str(BASE_MODEL).lower():
    raise ValueError(f'Tu choi D3 native SIMPLE_RADIAL model: {BASE_MODEL}')

base_parameters_path = BASE_MODEL / 'training_parameters.json'
base_parameters = (
    json.loads(base_parameters_path.read_text())
    if base_parameters_path.is_file() else {}
)
base_method = str(base_parameters.get('training_method', '')).lower()
if base_method and base_method != 'improvedgs':
    raise ValueError(f'Base model khong phai ImprovedGS: {base_method}')
for disabled_flag in ('coarse_to_fine', 'pose_aware_sampling'):
    if base_parameters.get(disabled_flag) is True:
        raise ValueError(f'Base model dang bat {disabled_flag}; can ImGS thuan tren main')

camera_json_path = BASE_MODEL / 'cameras.json'
if camera_json_path.is_file():
    camera_rows = json.loads(camera_json_path.read_text())
    camera_models = {
        str(row.get('camera_model', '')).upper() for row in camera_rows
    }
    if 'SIMPLE_RADIAL' in camera_models:
        raise ValueError('Tu choi checkpoint native SIMPLE_RADIAL; can model main PINHOLE')

BASE_BUDGET = int(base_parameters.get('budget', 5_500_000))
from vai.common import read_pose_rows, slice_pose_rows
all_pose_rows = read_pose_rows(SOURCE_PATH / 'test' / 'test_poses.csv')
selected_pose_rows = slice_pose_rows(
    all_pose_rows,
    start_index=POSE_START_INDEX,
    pose_count=POSE_COUNT,
)
selected_indices = list(range(POSE_START_INDEX, POSE_START_INDEX + len(selected_pose_rows)))
expected_indices = list(range(POSE_START_INDEX, min(POSE_END_INDEX, len(all_pose_rows))))
assert selected_indices == expected_indices
print('Base budget:', BASE_BUDGET)
print('Selected pose count:', len(selected_pose_rows), '/', len(all_pose_rows))
for index, row in zip(selected_indices, selected_pose_rows):
    print(f'  pose[{index:03d}] {row["image_name"]}')

In [ ]:
command = [
    sys.executable, '-u', 'vai_test_pose_finetune.py',
    '--source_path', str(SOURCE_PATH),
    '--base_model_path', str(BASE_MODEL),
    '--base_iteration', str(BASE_ITERATION),
    '--model_path', str(OUTPUT_MODEL_ROOT),
    '--scene_name', SCENE_NAME,
    '--pose_start_index', str(POSE_START_INDEX),
    '--pose_count', str(POSE_COUNT),
    '--fine_tune_steps', str(FINE_TUNE_STEPS),
    '--split_from_step', '0',
    '--split_until_step', str(SPLIT_UNTIL_STEP),
    '--top_k', str(TOP_K),
    '--sigma_multiplier', str(SIGMA_MULTIPLIER),
    '--save_pose_models', str(SAVE_POSE_MODELS).lower(),
    '--training_method', 'improvedgs',
    '--use_las', 'true',
    '--use_eas', 'true',
    '--use_rap', 'true',
    '--use_mu', 'true',
    '--coarse_to_fine', 'false',
    '--pose_aware_sampling', 'false',
    '--densify_grad_threshold', str(DENSIFY_GRAD_THRESHOLD),
    '--budget', str(BASE_BUDGET),
    '--resolution', '-1',
    '--data_device', 'cpu',
    '--eval', 'false',
    '--train_test_exp', 'false',
    '--output_root', str(RENDER_ROOT),
    '--output_extension', 'csv',
    '--save_png', str(SAVE_PNG).lower(),
    '--png_root', str(PNG_ROOT),
    '--evaluate', 'true',
    '--require_gt', 'true',
    '--overwrite', 'true',
    '--progress_bar_width', '100',
]
print(' '.join(command))
env = os.environ.copy()
env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
subprocess.run(command, cwd=REPO_DIR, check=True, env=env)

In [ ]:
# Validate va dong goi rieng batch 15 pose, khong doi code cho cac batch sau.
jpeg_zip = WORK_ROOT / f'{EXPERIMENT_NAME}_jpeg.zip'
subprocess.run([
    sys.executable, 'vai_package.py',
    '--phase_dir', str(PHASE_DIR),
    '--set_name', SET_NAME,
    '--submission_dir', str(RENDER_ROOT),
    '--zip_path', str(jpeg_zip),
    '--subset', SCENE_NAME,
    '--output_extension', 'csv',
    '--pose_start_index', str(POSE_START_INDEX),
    '--pose_count', str(POSE_COUNT),
], check=True)
print('JPEG ZIP:', jpeg_zip)

if SAVE_PNG:
    png_zip = WORK_ROOT / f'{EXPERIMENT_NAME}_png.zip'
    subprocess.run([
        sys.executable, 'vai_package.py',
        '--phase_dir', str(PHASE_DIR),
        '--set_name', SET_NAME,
        '--submission_dir', str(PNG_ROOT),
        '--zip_path', str(png_zip),
        '--subset', SCENE_NAME,
        '--output_extension', 'png',
        '--pose_start_index', str(POSE_START_INDEX),
        '--pose_count', str(POSE_COUNT),
    ], check=True)
    print('PNG ZIP:', png_zip)

eval_zip = WORK_ROOT / f'{EXPERIMENT_NAME}_eval.zip'
json_files = [
    OUTPUT_MODEL_ROOT / 'test_pose_finetune_manifest.json',
    OUTPUT_MODEL_ROOT / 'result_test.json',
    OUTPUT_MODEL_ROOT / 'vai_per_view.json',
]
json_files.extend(sorted((OUTPUT_MODEL_ROOT / 'poses').glob('*/view_selection.json')))
with zipfile.ZipFile(eval_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in json_files:
        if not path.is_file():
            raise FileNotFoundError(path)
        archive.write(path, path.relative_to(OUTPUT_MODEL_ROOT))
print('Eval ZIP:', eval_zip)

In [ ]:
manifest_path = OUTPUT_MODEL_ROOT / 'test_pose_finetune_manifest.json'
manifest = json.loads(manifest_path.read_text())
assert manifest['pose_indices'] == expected_indices
assert manifest['completed_pose_count'] == len(expected_indices)
assert all(pose['optimizer_updates'] == FINE_TUNE_STEPS for pose in manifest['poses'])
assert all(pose['selection']['selected_top_k'] == TOP_K for pose in manifest['poses'])
print('Completed:', manifest['completed_pose_count'], '/', manifest['source_test_pose_count'])
print('Pose indices:', manifest['pose_indices'])
print('Render dir:', manifest['render_dir'])
print('Evaluation:', json.dumps(manifest.get('evaluation', {}), indent=2))
print('First pose top view:', manifest['poses'][0]['selection']['views'][0])